# Hair App Canonical Crop V2 Test

이 노트북은 보완된 **crop v2만** 실행한다. Pixel3DMM, PIPNet, FaRL segmentation, normal/UV, FLAME fitting은 실행하지 않는다.

v2 변경점: 5개 RetinaFace point 보존, profile/불합리한 eye pair의 roll 차단, 다중 얼굴 주 피사체 선택, reflect padding과 observed-source validity mask, 경고 metadata. 사진은 경고가 있어도 버리지 않는다.

In [ ]:
!pip -q install 'git+https://github.com/FacePerceiver/facer.git@ddd35c76ff840174b8a5403ad1c1255e37b8782b'
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 1. Drive와 실험 경로

기존 v1 결과와 비교할 수 있도록 v2는 `crop_test_512_v2/`에 따로 저장한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
INPUT_DIR = Path('/content/drive/MyDrive/hair_app/inputs')
OUTPUT_ROOT = Path('/content/drive/MyDrive/hair_app/crop_test_512_v2')
OUTPUT_SIZE = 512
BBOX_MARGIN = 1.50
VERTICAL_CENTER_OFFSET = -0.04
DETECTION_THRESHOLD = 0.5

input_files = sorted(path for path in INPUT_DIR.iterdir() if path.suffix.lower() in {'.jpg', '.jpeg', '.png'})
print('inputs:', len(input_files), [path.name for path in input_files])
assert input_files, f'입력 이미지 없음: {INPUT_DIR}'

## 2. v2 엔진 준비

먼저 GitHub `main`에서 엔진을 받는다. 아직 commit/push 전이라면 파일 선택 창에서 `canonical_face_crop_v2.py`를 직접 올리면 된다.

In [ ]:
from urllib.request import urlretrieve
from google.colab import files

ENGINE_PATH = Path('/content/canonical_face_crop_v2.py')
ENGINE_URL = 'https://raw.githubusercontent.com/Leejuseop/hair_app/main/experiments/milestone1_geometry_bakeoff/canonical_face_crop_v2.py'
try:
    urlretrieve(ENGINE_URL, ENGINE_PATH)
    print('GitHub engine download: PASS')
except Exception as error:
    print('GitHub에서 받지 못했습니다:', error)
    print('canonical_face_crop_v2.py를 선택하세요.')
    uploaded = files.upload()
    assert 'canonical_face_crop_v2.py' in uploaded
    ENGINE_PATH.write_bytes(uploaded['canonical_face_crop_v2.py'])

assert ENGINE_PATH.exists() and ENGINE_PATH.stat().st_size > 0
print('crop v2 engine:', ENGINE_PATH, ENGINE_PATH.stat().st_size, 'bytes')

## 3. crop v2 실행

In [ ]:
import shutil
import subprocess
import sys

assert OUTPUT_ROOT.name == 'crop_test_512_v2', f'안전하지 않은 출력 경로: {OUTPUT_ROOT}'
shutil.rmtree(OUTPUT_ROOT, ignore_errors=True)
command = [
    sys.executable, str(ENGINE_PATH),
    '--input-dir', str(INPUT_DIR),
    '--output-root', str(OUTPUT_ROOT),
    '--output-size', str(OUTPUT_SIZE),
    '--bbox-margin', str(BBOX_MARGIN),
    '--vertical-center-offset', str(VERTICAL_CENTER_OFFSET),
    '--detection-threshold', str(DETECTION_THRESHOLD),
    '--device', DEVICE,
    '--clean-output',
]
subprocess.run(command, check=True)
print('CROP V2 RUN: PASS')

## 4. 원본 · v2 crop · validity mask 비교

원본의 초록 bbox가 선택된 얼굴이고 빨간 bbox는 선택되지 않은 다른 얼굴이다. crop의 점은 eyes/nose/mouth다. validity mask에서 흰색은 실제 원본 픽셀, 검은색은 검은 여백 대신 reflect로 채운 참고 영역이다.

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image, ImageOps

manifest = json.loads((OUTPUT_ROOT / 'crop_meta' / 'manifest.json').read_text(encoding='utf-8'))
fig, axes = plt.subplots(len(manifest['items']), 3, figsize=(15, 4.5 * len(manifest['items'])), squeeze=False)
for row, item in enumerate(manifest['items']):
    with Image.open(INPUT_DIR / item['source_name']) as source_file:
        source = ImageOps.exif_transpose(source_file).convert('RGB')
    crop = Image.open(OUTPUT_ROOT / 'cropped' / item['derived_name']).convert('RGB')
    validity = Image.open(OUTPUT_ROOT / 'crop_validity' / item['validity_mask_name']).convert('L')

    axes[row, 0].imshow(source)
    selected = item['face_selection']['selected_detector_index']
    for candidate in item['face_selection']['candidate_rankings']:
        x1, y1, x2, y2 = candidate['bbox_xyxy']
        color = 'lime' if candidate['detector_index'] == selected else 'red'
        axes[row, 0].add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, color=color, linewidth=2))
    for point in item['observation']['landmarks5_xy']:
        axes[row, 0].scatter(point[0], point[1], s=20, c='cyan')
    axes[row, 0].set_title(f"원본: {item['source_name']} | faces={item['face_selection']['candidate_count']}")

    axes[row, 1].imshow(crop)
    transformed = np.asarray(item['transformed_landmarks5'])
    axes[row, 1].scatter(transformed[:, 0], transformed[:, 1], s=24, c=['cyan','cyan','yellow','magenta','magenta'])
    axes[row, 1].set_title(
        f"crop: {item['derived_name']} | raw/applied roll={item['raw_eye_roll_degrees']:.1f}/{item['roll_degrees_applied']:.1f}°\n"
        f"warnings={item['warnings']}"
    )

    axes[row, 2].imshow(validity, cmap='gray', vmin=0, vmax=255)
    axes[row, 2].set_title(f"observed={item['observed_source_fraction']:.3f}")
    for axis in axes[row]:
        axis.axis('off')
plt.tight_layout()
plt.show()

## 5. 자동 계약 검사

자동 검사는 파일 수·크기·행렬·mask만 확인한다. 얼굴 coverage와 잘못된 얼굴 선택은 위 그림에서 사람이 최종 확인한다.

In [ ]:
crop_files = sorted((OUTPUT_ROOT / 'cropped').glob('*.jpg'))
mask_files = sorted((OUTPUT_ROOT / 'crop_validity').glob('*.png'))
meta_files = sorted(path for path in (OUTPUT_ROOT / 'crop_meta').glob('*.json') if path.name != 'manifest.json')
assert len(input_files) == len(crop_files) == len(mask_files) == len(meta_files) == manifest['count']
for crop_path, mask_path, item in zip(crop_files, mask_files, manifest['items']):
    assert Image.open(crop_path).size == (512, 512)
    assert Image.open(mask_path).size == (512, 512)
    forward = np.asarray(item['source_to_crop'])
    inverse = np.asarray(item['crop_to_source'])
    assert np.allclose(inverse @ forward, np.eye(3), atol=1e-6)
    assert 0.0 <= item['observed_source_fraction'] <= 1.0
print('CROP V2 CONTRACT: PASS')
print('이제 위 8개 원본/crop/mask를 v1 결과와 비교하세요. 아직 Pixel3DMM 다음 단계는 실행하지 않습니다.')